In [1]:
import gc
from itertools import combinations
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import friedmanchisquare, studentized_range
from sklearn.base import clone
from sklearn.metrics import f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate

ROOT = Path(r"C:\tcc\network-ids-ml-generalization")
DATA_DIR = ROOT / "datasets" / "final" / "ml"
MODEL_DIR = ROOT / "models" / "randomized_search"
OUTPUT_DIR = ROOT / "results" / "friedman_test"
FOLD_RESULTS_DIR = OUTPUT_DIR / "folds"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FOLD_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 1
N_SPLITS = 5
N_REPEATS = 30
N_JOBS = -1
ALPHA = 0.05
PRIMARY_METRIC = "f1_macro"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


In [2]:
TRAINING_CONFIGS = {
    "cicids2017": ["cicids2017"],
    "unsw_nb15": ["unsw_nb15"],
    "iot23": ["iot23"],
    "cicids2017_unsw_nb15": ["cicids2017", "unsw_nb15"],
    "cicids2017_iot23": ["cicids2017", "iot23"],
    "unsw_nb15_iot23": ["unsw_nb15", "iot23"],
}

MODEL_ORDER = [
    "decision_tree",
    "random_forest",
    "xgboost",
    "linear_svm",
    "mlp",
]

TRAINING_SOURCES_TO_RUN = [
    "cicids2017",
    "unsw_nb15",
    "iot23",
]
MODELS_TO_RUN = MODEL_ORDER.copy()

#pos_label=0 benigno ; pos_label=1 ataque.
#zero_division=0. se a métrica envolver uma divisão por zero, registre o resultado como zero.
scoring = {
    "precision_attack": make_scorer(precision_score, pos_label=1, zero_division=0),
    "recall_attack": make_scorer(recall_score, pos_label=1, zero_division=0),
    "f1_attack": make_scorer(f1_score, pos_label=1, zero_division=0),
    "precision_benign": make_scorer(precision_score, pos_label=0, zero_division=0),
    "recall_benign": make_scorer(recall_score, pos_label=0, zero_division=0),
    "f1_benign": make_scorer(f1_score, pos_label=0, zero_division=0),
    "f1_macro": make_scorer(
        f1_score,
        average="macro",
        labels=[0, 1],
        zero_division=0,
    ),
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
}


In [3]:
model_files = {
    (model_name, training_source): (
        MODEL_DIR / model_name / f"{training_source}.joblib"
    )
    for model_name in MODELS_TO_RUN
    for training_source in TRAINING_SOURCES_TO_RUN
}

In [4]:
# load na combinação (modelo, base)| recuperar colunas | limpar objetos
reference_pipeline = joblib.load(model_files[(MODEL_ORDER[0], TRAINING_SOURCES_TO_RUN[0])])
preprocessor = reference_pipeline.named_steps["preprocessor"]
numerical_features = list(preprocessor.transformers[0][2])
categorical_features = list(preprocessor.transformers[1][2])
feature_columns = numerical_features + categorical_features
del reference_pipeline, preprocessor

repeated_cross_validation = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
)

def load_training_data(dataset_names):
    columns_to_load = feature_columns + ["label_binary", "dataset_name"]
    dataframes = [
        pd.read_parquet(
            DATA_DIR / f"{dataset_name}_train.parquet",
            columns=columns_to_load,
        )
        for dataset_name in dataset_names
    ]
    training_df = pd.concat(dataframes, ignore_index=True)
    features = training_df[feature_columns]
    target = training_df["label_binary"].eq("ATTACK").astype("int8")

    #Estratificação por base e classe
    stratification_labels = (
        training_df["dataset_name"].astype("string")
        + "__"
        + target.astype("string")
    )
    # gera e fixa 30 repetições de 5 folds estratificados (150 pares de treino e teste).
    cv_splits = list(
        repeated_cross_validation.split(features, stratification_labels)
    )
    return features, target, cv_splits

#monta o caminho do csv com os 150 resultados de cada combinação
def fold_result_path(model_name, training_source):
    model_output_dir = FOLD_RESULTS_DIR / model_name
    model_output_dir.mkdir(parents=True, exist_ok=True)
    return model_output_dir / f"{training_source}.csv"

# verifica se o resultado está completo para evitar uma nova execução.
def result_is_complete(result_path):
    if not result_path.exists():
        return False
    result = pd.read_csv(result_path)
    expected_rows = N_SPLITS * N_REPEATS
    required_columns = {
        "model",
        "training_source",
        "repeat",
        "fold",
        "n_splits",
        "n_repeats",
        "random_state",
        PRIMARY_METRIC,
    }
    if len(result) != expected_rows:
        return False
    if not required_columns.issubset(result.columns):
        return False
    # confirma que o arquivo corresponde à configuração atual e não possui resultados inválidos
    return (
        result["n_splits"].eq(N_SPLITS).all()
        and result["n_repeats"].eq(N_REPEATS).all()
        and result["random_state"].eq(RANDOM_STATE).all()
        and not result[["repeat", "fold"]].duplicated().any()
        and not result[PRIMARY_METRIC].isna().any()
    )

#avalia um modelo com seus melhores hiperparâmetros nas 30 repetições do k-fold
def evaluate_selected_model(model_name, training_source, features, target, cv_splits):

    #clone mantém a configuração do modelo, mas remove o treinamento anterior
    selected_pipeline = clone(joblib.load(model_files[(model_name, training_source)]))
    evaluation_start = perf_counter()
    # executa os 150 pares de treino e teste
    cv_results = cross_validate(
        selected_pipeline,
        features,
        target,
        scoring=scoring,
        cv=cv_splits,
        n_jobs=N_JOBS,
        return_train_score=False,
        error_score="raise",
    )
    elapsed_seconds = perf_counter() - evaluation_start
    #organiza cada resultado pelo número da repetição e do fold
    result = pd.DataFrame({
        "model": model_name,
        "training_source": training_source,
        "repeat": np.repeat(np.arange(1, N_REPEATS + 1), N_SPLITS),
        "fold": np.tile(np.arange(1, N_SPLITS + 1), N_REPEATS),
        "n_splits": N_SPLITS,
        "n_repeats": N_REPEATS,
        "random_state": RANDOM_STATE,
        "fit_time_seconds": cv_results["fit_time"],
        "score_time_seconds": cv_results["score_time"],
    })
    # adiciona ao resultado as métricas calculadas em cada fold
    for metric_name in scoring:
        result[metric_name] = cv_results[f"test_{metric_name}"]
    result["evaluation_elapsed_seconds"] = elapsed_seconds
    return result


In [5]:
# chama as funções da CV
total_start = perf_counter()
completed_experiments = 0
total_experiments = len(TRAINING_SOURCES_TO_RUN) * len(MODELS_TO_RUN)

for training_source in TRAINING_SOURCES_TO_RUN:
    pending_models = [
        model_name
        for model_name in MODELS_TO_RUN
        if not result_is_complete(fold_result_path(model_name, training_source))
    ]

    if pending_models:
        features, target, cv_splits = load_training_data(
            TRAINING_CONFIGS[training_source]
        )

        for model_name in MODELS_TO_RUN:
            result_path = fold_result_path(model_name, training_source)
            completed_experiments += 1

            if result_is_complete(result_path):
                print(
                    f"{completed_experiments:02d}/{total_experiments} - "
                    f"{model_name} - {training_source} - já concluído"
                )
                continue

            result = evaluate_selected_model(
                model_name,
                training_source,
                features,
                target,
                cv_splits,
            )
            result.to_csv(result_path, index=False)
            elapsed_minutes = result["evaluation_elapsed_seconds"].iloc[0] / 60
            print(
                f"{completed_experiments:02d}/{total_experiments} - "
                f"{model_name} - {training_source} - "
                f"{elapsed_minutes:.2f} min"
            )

        del features, target, cv_splits
        gc.collect()
    else:
        completed_experiments += len(MODELS_TO_RUN)
        print(f"{training_source} - todos os modelos já concluídos")

total_elapsed_seconds = perf_counter() - total_start
print(f"tempo desta execução: {total_elapsed_seconds / 60:.2f} min")


cicids2017 - todos os modelos já concluídos
unsw_nb15 - todos os modelos já concluídos
iot23 - todos os modelos já concluídos
tempo desta execução: 0.00 min


In [6]:
# dfs carregados dos csvs individuais de cada modelo e origem
fold_frames = [
    pd.read_csv(fold_result_path(model_name, training_source))
    for training_source in TRAINING_SOURCES_TO_RUN
    for model_name in MODELS_TO_RUN
]

# concatena e organiza os resultados dos folds
fold_results = pd.concat(fold_frames, ignore_index=True)
fold_results = fold_results.sort_values(
    ["training_source", "model", "repeat", "fold"]
).reset_index(drop=True)

# define como agregar os 5 folds: média das métricas e soma dos tempos
aggregation = {metric_name: "mean" for metric_name in scoring}
aggregation.update({
    "fit_time_seconds": "sum",
    "score_time_seconds": "sum",
})

# agrupa os 5 folds de cada repetição por origem e modelo
# aplica a média às métricas e a soma aos tempos por repetição
repeat_results = (
    fold_results.groupby(
        ["training_source", "model", "repeat"],
        as_index=False,
    )
    .agg(aggregation)
)
# salva 30 repetições para cada combinação de origem e modelo
repeat_results.to_csv(OUTPUT_DIR / "repeat_results.csv", index=False)


In [7]:
# para cada origem, Friedman compara os 5 modelos em 30 repetições pareadas
# cada valor comparado é o F1-macro médio dos 5 folds daquela repetição
# o teste transforma esses valores em rankings e verifica se os desempenhos são equivalentes

# armazena os resultados e as matrizes de cada origem
friedman_rows = []
friedman_pivots = {}

for training_source in TRAINING_SOURCES_TO_RUN:
    # seleciona os 150 resultados da origem atual
    source_results = repeat_results.loc[
        repeat_results["training_source"].eq(training_source)
    ]
    # organiza as 30 repetições nas linhas e os 5 modelos nas colunas
    pivot = source_results.pivot(
        index="repeat",
        columns="model",
        values=PRIMARY_METRIC,
    ).reindex(columns=MODEL_ORDER)

    # confirma que a matriz está completa
    assert pivot.shape == (N_REPEATS, len(MODEL_ORDER))
    assert not pivot.isna().any().any()

    # executa Friedman com as 30 médias de cada modelo
    statistic, p_value = friedmanchisquare(
        *[pivot[model_name].to_numpy() for model_name in MODEL_ORDER]
    )
    # calcula a posição média de cada modelo nas 30 repetições
    ranks = pivot.rank(axis=1, ascending=False, method="average")
    row = {
        "training_source": training_source,
        "statistic": statistic,
        "p_value": p_value,
    }
    for model_name in MODEL_ORDER:
        row[f"mean_rank_{model_name}"] = ranks[model_name].mean()
    friedman_rows.append(row)
    friedman_pivots[training_source] = pivot

friedman_results = pd.DataFrame(friedman_rows)
friedman_results.to_csv(OUTPUT_DIR / "friedman_results.csv", index=False)

display(friedman_results[["training_source", "statistic", "p_value"]])


,training_source,statistic,p_value
0,cicids2017,114.960000,6.364445e-24
1,unsw_nb15,115.306667,5.367452e-24
2,iot23,116.906667,2.444647e-24


In [8]:
# compara todos os pares de modelos pelo pós-hoc de Nemenyi
posthoc_rows = []

for friedman_result in friedman_results.itertuples(index=False):
    if friedman_result.p_value >= ALPHA:
        continue

    pivot = friedman_pivots[friedman_result.training_source]
    ranks = pivot.rank(axis=1, ascending=False, method="average")
    mean_ranks = ranks.mean()
    n_blocks = len(ranks)
    n_models = len(MODEL_ORDER)
    standard_error = np.sqrt(n_models * (n_models + 1) / (6 * n_blocks))
    critical_difference = (
        studentized_range.isf(ALPHA, n_models, np.inf)
        / np.sqrt(2)
        * standard_error
    )

    for model_a, model_b in combinations(MODEL_ORDER, 2):
        mean_rank_a = mean_ranks[model_a]
        mean_rank_b = mean_ranks[model_b]
        rank_difference = abs(mean_rank_a - mean_rank_b)
        statistic = np.sqrt(2) * rank_difference / standard_error
        p_value = studentized_range.sf(statistic, n_models, np.inf)
        if mean_rank_a < mean_rank_b:
            better_model = model_a
        elif mean_rank_b < mean_rank_a:
            better_model = model_b
        else:
            better_model = "tie"

        posthoc_rows.append({
            "training_source": friedman_result.training_source,
            "metric": PRIMARY_METRIC,
            "model_a": model_a,
            "model_b": model_b,
            "mean_rank_a": mean_rank_a,
            "mean_rank_b": mean_rank_b,
            "rank_difference": rank_difference,
            "standard_error": standard_error,
            "critical_difference": critical_difference,
            "statistic": statistic,
            "p_value": p_value,
            "significant": p_value < ALPHA,
            "better_model": better_model,
        })

posthoc_columns = [
    "training_source",
    "metric",
    "model_a",
    "model_b",
    "mean_rank_a",
    "mean_rank_b",
    "rank_difference",
    "standard_error",
    "critical_difference",
    "statistic",
    "p_value",
    "significant",
    "better_model",
]
posthoc_results = pd.DataFrame(posthoc_rows, columns=posthoc_columns)
posthoc_results.to_csv(OUTPUT_DIR / "posthoc_nemenyi.csv", index=False)

if posthoc_results.empty:
    print("nenhum pós-hoc foi necessário")
else:
    display(posthoc_results)


,training_source,metric,model_a,model_b,mean_rank_a,mean_rank_b,rank_difference,standard_error,critical_difference,statistic,p_value,significant,better_model
0,cicids2017,f1_macro,decision_tree,random_forest,3.700000,1.000000,2.700000,0.408248,1.113609,9.353074,3.749161e-10,True,random_forest
1,cicids2017,f1_macro,decision_tree,xgboost,3.700000,2.000000,1.700000,0.408248,1.113609,5.888973,3.018981e-04,True,xgboost
2,cicids2017,f1_macro,decision_tree,linear_svm,3.700000,5.000000,1.300000,0.408248,1.113609,4.503332,1.262753e-02,True,decision_tree
3,cicids2017,f1_macro,decision_tree,mlp,3.700000,3.300000,0.400000,0.408248,1.113609,1.385641,8.644775e-01,False,mlp
4,cicids2017,f1_macro,random_forest,xgboost,1.000000,2.000000,1.000000,0.408248,1.113609,3.464102,1.024489e-01,False,random_forest
5,cicids2017,f1_macro,random_forest,linear_svm,1.000000,5.000000,4.000000,0.408248,1.113609,13.856406,1.110223e-16,True,random_forest
6,cicids2017,f1_macro,random_forest,mlp,1.000000,3.300000,2.300000,0.408248,1.113609,7.967434,1.758120e-07,True,random_forest
7,cicids2017,f1_macro,xgboost,linear_svm,2.000000,5.000000,3.000000,0.408248,1.113609,10.392305,2.004952e-12,True,xgboost
8,cicids2017,f1_macro,xgboost,mlp,2.000000,3.300000,1.300000,0.408248,1.113609,4.503332,1.262753e-02,True,xgboost
9,cicids2017,f1_macro,linear_svm,mlp,5.000000,3.300000,1.700000,0.408248,1.113609,5.888973,3.018981e-04,True,mlp


In [9]:
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display

RESULTS_ROOT = Path(r"C:\tcc\network-ids-ml-generalization\results\friedman_test")
PRESENTATION_DIR = RESULTS_ROOT / "presentation"
PRESENTATION_DIR.mkdir(parents=True, exist_ok=True)

REPEAT_RESULTS_FILE = RESULTS_ROOT / "repeat_results.csv"
FRIEDMAN_RESULTS_FILE = RESULTS_ROOT / "friedman_results.csv"
POSTHOC_RESULTS_FILE = RESULTS_ROOT / "posthoc_nemenyi.csv"

for result_file in [
    REPEAT_RESULTS_FILE,
    FRIEDMAN_RESULTS_FILE,
    POSTHOC_RESULTS_FILE,
]:
    if not result_file.exists():
        raise FileNotFoundError(f"Resultado calculado não encontrado: {result_file}")

saved_repeat_results = pd.read_csv(REPEAT_RESULTS_FILE)
saved_friedman_results = pd.read_csv(FRIEDMAN_RESULTS_FILE)
saved_posthoc_results = pd.read_csv(POSTHOC_RESULTS_FILE)

assert len(saved_repeat_results) == 450
assert len(saved_friedman_results) == 3
assert len(saved_posthoc_results) <= 30
assert len(saved_posthoc_results) % 10 == 0
assert saved_repeat_results.groupby(["training_source", "model"]).size().eq(30).all()

MODEL_ORDER = ["decision_tree", "random_forest", "xgboost", "linear_svm", "mlp"]
TRAINING_SOURCE_ORDER = [
    "cicids2017",
    "unsw_nb15",
    "iot23",
]
MODEL_LABELS = {
    "decision_tree": "Árvore de decisão",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
    "linear_svm": "SVM linear",
    "mlp": "MLP",
}
SOURCE_LABELS = {
    "cicids2017": "CICIDS2017",
    "unsw_nb15": "UNSW-NB15",
    "iot23": "IoT-23",
    "cicids2017_unsw_nb15": "CICIDS2017 + UNSW-NB15",
    "cicids2017_iot23": "CICIDS2017 + IoT-23",
    "unsw_nb15_iot23": "UNSW-NB15 + IoT-23",
}
MODEL_COLORS = {
    "Árvore de decisão": "#4C78A8",
    "Random Forest": "#F58518",
    "XGBoost": "#54A24B",
    "SVM linear": "#E45756",
    "MLP": "#B279A2",
}

loaded_results = pd.DataFrame({
    "arquivo": [
        REPEAT_RESULTS_FILE.name,
        FRIEDMAN_RESULTS_FILE.name,
        POSTHOC_RESULTS_FILE.name,
    ],
    "linhas": [
        len(saved_repeat_results),
        len(saved_friedman_results),
        len(saved_posthoc_results),
    ],
    "status": "carregado",
})
display(loaded_results)


,arquivo,linhas,status
0,repeat_results.csv,450,carregado
1,friedman_results.csv,3,carregado
2,posthoc_nemenyi.csv,30,carregado


In [10]:
boxplot_data = saved_repeat_results.copy()
boxplot_data["Modelo"] = boxplot_data["model"].map(MODEL_LABELS)

BOXPLOT_DIR = PRESENTATION_DIR / "boxplots"
BOXPLOT_DIR.mkdir(parents=True, exist_ok=True)

for training_source in TRAINING_SOURCE_ORDER:
    source_data = boxplot_data.loc[
        boxplot_data["training_source"].eq(training_source)
    ]
    boxplot_figure = px.box(
        source_data,
        x="Modelo",
        y="f1_macro",
        color="Modelo",
        points=False,
        color_discrete_map=MODEL_COLORS,
        category_orders={
            "Modelo": [MODEL_LABELS[model] for model in MODEL_ORDER],
        },
        labels={"f1_macro": "F1-macro"},
        title=(
            f"Distribuição do F1-macro — {SOURCE_LABELS[training_source]}"
            "<br><sup>30 repetições; cada valor é a média dos cinco folds</sup>"
        ),
    )
    boxplot_figure.update_traces(line={"width": 1.5})
    boxplot_figure.update_yaxes(range=[0.80, 1.001])
    boxplot_figure.update_layout(
        template="plotly_white",
        width=1100,
        height=650,
        showlegend=False,
    )
    boxplot_figure.write_image(
        BOXPLOT_DIR / f"f1_macro_{training_source}.png",
        scale=2,
    )
    display(boxplot_figure)


In [11]:
friedman_table = saved_friedman_results.rename(columns={
    "mean_rank_decision_tree": "Ranking - Árvore de decisão",
    "mean_rank_random_forest": "Ranking - Random Forest",
    "mean_rank_xgboost": "Ranking - XGBoost",
    "mean_rank_linear_svm": "Ranking - SVM linear",
    "mean_rank_mlp": "Ranking - MLP",
    "statistic": "Estatística de Friedman",
    "p_value": "p-valor",
}).copy()
friedman_table["Base"] = friedman_table["training_source"].map(SOURCE_LABELS)
rank_columns = [f"Ranking - {MODEL_LABELS[model]}" for model in MODEL_ORDER]
friedman_table[rank_columns] = friedman_table[rank_columns].round(2)
friedman_table["Estatística de Friedman"] = (
    friedman_table["Estatística de Friedman"].round(2)
)
friedman_table["p-valor"] = friedman_table["p-valor"].map(
    lambda value: f"{value:.2e}"
)
friedman_table = friedman_table[[
    "Base",
    *rank_columns,
    "Estatística de Friedman",
    "p-valor",
]]
friedman_table.to_csv(PRESENTATION_DIR / "friedman_table.csv", index=False)
display(friedman_table)

nemenyi_summary_rows = []
for training_source in TRAINING_SOURCE_ORDER:
    friedman_row = saved_friedman_results.loc[
        saved_friedman_results["training_source"].eq(training_source)
    ].iloc[0]
    ordered_models = sorted(
        MODEL_ORDER,
        key=lambda model: friedman_row[f"mean_rank_{model}"],
    )
    source_posthoc = saved_posthoc_results.loc[
        saved_posthoc_results["training_source"].eq(training_source)
    ]
    ranking_order = "; ".join(
        f"{position}º {MODEL_LABELS[model]} "
        f"({friedman_row[f'mean_rank_{model}']:.2f})"
        for position, model in enumerate(ordered_models, start=1)
    )
    non_significant_pairs = [
        f"{MODEL_LABELS[comparison.model_a]} × {MODEL_LABELS[comparison.model_b]}"
        for comparison in source_posthoc.itertuples(index=False)
        if not comparison.significant
    ]

    nemenyi_summary_rows.append({
        "Base": SOURCE_LABELS[training_source],
        "Ordem por ranking médio": ranking_order,
        "Pares sem diferença significativa": "; ".join(non_significant_pairs),
    })

nemenyi_summary = pd.DataFrame(nemenyi_summary_rows)
nemenyi_summary.to_csv(PRESENTATION_DIR / "nemenyi_summary.csv", index=False)
display(nemenyi_summary)


,Base,Ranking - Árvore de decisão,Ranking - Random Forest,Ranking - XGBoost,Ranking - SVM linear,Ranking - MLP,Estatística de Friedman,p-valor
0,CICIDS2017,3.7,1.00,2.00,5.0,3.30,114.96,6.36e-24
1,UNSW-NB15,3.0,1.27,1.73,5.0,4.00,115.31,5.37e-24
2,IoT-23,3.0,2.07,1.00,5.0,3.93,116.91,2.44e-24


,Base,Ordem por ranking médio,Pares sem diferença significativa
0,CICIDS2017,1º Random Forest (1.00); 2º XGBoost (2.00); 3º MLP (3.30); 4º Árvore de decisão (3.70); 5º SVM linear (5.00),Árvore de decisão × MLP; Random Forest × XGBoost
1,UNSW-NB15,1º Random Forest (1.27); 2º XGBoost (1.73); 3º Árvore de decisão (3.00); 4º MLP (4.00); 5º SVM linear (5.00),Árvore de decisão × MLP; Random Forest × XGBoost; SVM linear × MLP
2,IoT-23,1º XGBoost (1.00); 2º Random Forest (2.07); 3º Árvore de decisão (3.00); 4º MLP (3.93); 5º SVM linear (5.00),Árvore de decisão × Random Forest; Árvore de decisão × MLP; Random Forest × XGBoost; SVM linear × MLP
